In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxIBTL)

This notebook processes and standardizes the **ToxIBTL** dataset, which provides peptide sequences annotated for toxicity and distributed across multiple file formats. The original data include both FASTA files with embedded labels and CSV tables, which are unified here into a single, consistent dataset.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxIBTL
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Parses raw ToxIBTL data from heterogeneous sources**:
  - FASTA files (`.fa`) with labels encoded in sequence identifiers,
  - CSV files containing explicit sequence–label annotations.
- **Integrates all sources into a unified dataset** with standardized columns (`sequence`, `label`).
- **Performs duplicate sequence analysis**:
  - consolidates identical sequences with consistent labels,
  - flags conflicting annotations as erroneous sequences.
- **Generates structured metadata** based on a centralized raw data description file.
- **Exports curated outputs** for downstream modeling and benchmarking:
  - `processed_toxic_dataset.csv`
  - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "ToxIBTL"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*.fa"):
    df = read_fasta_with_strange_character(file)
    df["label"] = df["id"].str.split("\t").str[1].astype(int)
    dfs.append(df)
df_fasta = pd.concat(dfs, ignore_index=True)

In [4]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*.csv"):
    df = pd.read_csv(file)
    dfs.append(df)
df_csv = pd.concat(dfs, ignore_index=True)

- Concatenate dataset

In [5]:
df_toxibtl = (
    pd.concat([df_fasta, df_csv], 
              ignore_index=True)
    [["sequence", "label"]]
)
df_toxibtl.shape

(25490, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_toxibtl, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(14092, 2)

In [8]:
df_errors.shape

(5, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_toxibtl)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2022,
 'last update date': datetime.datetime(2022, 1, 24, 0, 0),
 'download date': Timestamp('2025-04-01 00:00:00'),
 'file format': 'csv;fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/WLYLab/ToxIBTL/tree/main/Data',
 'publication': 'https://academic.oup.com/bioinformatics/article/38/6/1514/6499265?login=false',
 'number_of_raw_sequences': 25490,
 'number_of_sequences_retained': 14092,
 'number_of_positive_sequences': 5698,
 'number_of_negative_sequences': 8394,
 'number_of_erroneous_sequences': 5,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)